[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Hyperparameter Search


## What you will be able to do

Let scikit-learn try many settings of a model by cross-validation, read which one won and by how
much, and measure honestly how well the winner will do on stations nobody has seen.

## The idea

### The problem

A decision tree that predicts snow on 1 May has settings that change what it learns: how many
questions deep it may go, and how few stations it may leave at the end of a branch. A shallow tree
misses real patterns and a deep one memorizes its training stations, and nothing in the data says in
advance which depth is right. The only way to choose is to try several and compare them.

Trying them by hand means a loop over depths, a cross-validation for each, and code to keep track of
the best. Add a second setting and the loop has a loop inside it, and add a third and the code is
mostly bookkeeping. And there is a subtler problem in the comparison itself. Each cross-validated
score is an estimate with some noise in it, and picking the highest of eighteen noisy estimates picks
partly for the noise, so the winner's score is a little better than the winner.

### What a hyperparameter search is

> A **hyperparameter** is a setting chosen before a model is fitted, such as a tree's `max_depth`, as
> opposed to what `fit` learns. **`GridSearchCV`** fits and cross-validates a model at every
> combination of the values it is given, keeps the combination with the best mean score as
> `best_params_`, and refits the model with it on all the training data. **`RandomizedSearchCV`**
> tries a fixed number of combinations drawn at random from ranges. **Nested cross-validation** runs
> a whole search inside each fold of an outer cross-validation, to estimate how well the search's
> winner does on data the search never saw.

### Why it works that way

A search is an estimator. It has `fit`, `predict` and `score`, and once fitted it behaves like the
best model it found, so it fits anywhere a model does, inside `cross_val_score` included. That is what
makes nested cross-validation one line. The outer folds hold data back from the whole search, the
search picks its settings on the rest, and the outer score is measured on rows that played no part in
the choice.

The settings are named as `get_params` names them, so a model inside a pipeline is reached through its
step's name, `logisticregression__C`. The score a search maximizes is accuracy for a classifier unless
`scoring` names another, and the best combination is only as good as the measure it was chosen by.

A grid tries every combination, and its cost multiplies with each setting added. A random search
spends a fixed budget, and covers wide ranges for the same cost. Either way the winner can sit at the
edge of what was tried, and when the scores are still rising toward that edge, the range was too
narrow in that direction.

### Where this shows up

Nearly every model in scikit-learn has settings worth searching. The **Cross-Validation**
notebook supplies the folds, the **Pipelines** notebook the step names, and the **Imbalanced Classes**
notebook the reason to choose `scoring` with care. The **A Complete Workflow** notebook uses a small
search as one step of a whole project.

### What this notebook covers

- a search written by hand, and the same search with `GridSearchCV`
- `best_params_`, `best_score_`, `cv_results_`, and the refitted model
- a search over a pipeline's settings, by the step's name
- `RandomizedSearchCV` over ranges, with a fixed seed
- a search that chooses by recall instead of accuracy
- nested cross-validation, and three errors in how a search is set up and read

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

X = [[n % 12] for n in range(48)]
y = [n % 12 in (3, 4, 5, 9, 10, 11) for n in range(48)]

search = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [1, 2, 3]}, cv=4)
search.fit(X, y)
print(search.cv_results_["mean_test_score"].round(3))
print(search.best_params_, round(search.best_score_, 3))
```

```
[0.75 0.75 1.  ]
{'max_depth': 3} 1.0
```

A tree takes the best single question first, and on this pattern it needs a depth of 3 to separate the
bands, which is the depth the search found. It cross-validated a tree at each depth, four folds
each, recorded the mean score of each, and kept the best.

## Setup

Two libraries for the data, scikit-learn and SciPy, two models, and the tools that search.

- `numpy` and `pandas` build the stations and the tables of results, and the cell prints
  scikit-learn's version
- `train_test_split`, `StratifiedKFold` and `cross_val_score` split the stations and score a model
- `DecisionTreeClassifier`, and a pipeline of `StandardScaler` and `LogisticRegression` made with
  `make_pipeline`, are the models whose settings are searched
- `GridSearchCV` and `RandomizedSearchCV` are the searches
- `randint` from SciPy is a range of whole numbers for the random search to draw from

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. The column
predicted here is `snow_on_may_1`, and `FEATURES` names the five numeric columns it is predicted from.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import randint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### A search written by hand

The training set is three quarters of the stations, and every fold below is drawn from it. `folds`
fixes the five folds, so every setting is scored on the same ones:

In [2]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=0)

for depth in [2, 3, 4, 5, 6, 8]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    print(depth, round(cross_val_score(tree, X_train, y_train, cv=folds).mean(), 3))

2 0.748
3 0.812
4 0.829
5 0.823
6 0.818
8 0.782


A depth of 4 scores best, 0.829, with 5 and 6 close behind at 0.823 and 0.818, 3 a little further back
at 0.812, and 2 and 8 clearly worse. That is one setting, and the loop already needs its own
bookkeeping to remember the winner.

### The same search with GridSearchCV

`GridSearchCV` takes the model, a dictionary of settings and the values to try, and the folds. Here it
tries six depths and three values of `min_samples_leaf`, eighteen combinations, each fitted five times:

In [3]:
grid = {"max_depth": [2, 3, 4, 5, 6, 8], "min_samples_leaf": [1, 5, 20]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=folds)
search.fit(X_train, y_train)

print("best settings:", search.best_params_)
print("best mean score:", round(search.best_score_, 3))

best settings: {'max_depth': 4, 'min_samples_leaf': 5}
best mean score: 0.83


`best_params_` holds the combination with the highest mean score, the first in the grid's order when
several share it, and `best_score_` that mean across the five folds.

### Every combination in cv_results_

`cv_results_` keeps the score of every combination. As a table, sorted by rank with a stable sort so
that tied combinations keep the grid's order:

In [4]:
results = pd.DataFrame(search.cv_results_)
columns = ["param_max_depth", "param_min_samples_leaf", "mean_test_score", "std_test_score", "rank_test_score"]

results[columns].sort_values("rank_test_score", kind="stable").head(8).round(3)

,param_max_depth,param_min_samples_leaf,mean_test_score,std_test_score,rank_test_score
7,4,5,0.830,0.020,1
11,5,20,0.830,0.022,1
14,6,20,0.830,0.022,1
17,8,20,0.830,0.022,1
6,4,1,0.829,0.019,5
8,4,20,0.828,0.026,6
9,5,1,0.823,0.018,7
12,6,1,0.818,0.023,8


Four combinations tie for first at 0.830, and `best_params_` is the first of them in the grid's order.
The top six differ by 0.002 at most, and their standard deviations across folds are about 0.02, so
those six are equally good as far as five folds can tell.

### The refitted model

After the search, `GridSearchCV` refits the best combination on the whole training set, and the search
object predicts and scores with it:

In [5]:
print(search.best_estimator_)
print("test accuracy:", round(search.score(X_test, y_test), 3))

DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0)
test accuracy: 0.823


On the 300 test stations the tree scores 0.823, a little below its cross-validated 0.83.

### Naming a pipeline's settings

A pipeline's settings are named after its steps, two underscores and the setting's own name:

In [6]:
pipeline = make_pipeline(StandardScaler(), LogisticRegression())
print([name for name in pipeline.get_params() if name.startswith("logisticregression__")][:3])

logistic_search = GridSearchCV(pipeline, {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}, cv=folds)
logistic_search.fit(X_train, y_train)

logistic_results = pd.DataFrame(logistic_search.cv_results_)
print("best settings:", logistic_search.best_params_)
print("mean scores:", logistic_results["mean_test_score"].round(3).tolist())
print("stations right of 900:", (logistic_results["mean_test_score"] * len(X_train)).round().astype(int).tolist())
print("test accuracy:", round(logistic_search.score(X_test, y_test), 3))

['logisticregression__C', 'logisticregression__class_weight', 'logisticregression__dual']
best settings: {'logisticregression__C': 1}
mean scores: [0.833, 0.849, 0.854, 0.853, 0.853]
stations right of 900: [750, 764, 769, 768, 768]
test accuracy: 0.84


`C` controls how strongly the logistic regression is held back from fitting its training rows, and
smaller values hold it back more. The scores rise from 0.833 at 0.01 to 0.854 at 1 and stay level
after. `C=1` leads 10 and 100 by one station of the 900 the folds score, so the three are tied in
practice, and `best_params_` holds the highest mean. Cross-validated, the logistic regression's 0.854
is above the tree's 0.83, and on the test set it scores 0.84.

### A random search over ranges

`RandomizedSearchCV` draws `n_iter` combinations from distributions. `randint(2, 12)` gives whole
numbers from 2 to 11, and `random_state` fixes the draws:

In [7]:
ranges = {"max_depth": randint(2, 12), "min_samples_leaf": randint(1, 40)}
random_search = RandomizedSearchCV(DecisionTreeClassifier(random_state=0), ranges, n_iter=15, cv=folds,
                                   random_state=0)
random_search.fit(X_train, y_train)

tied = pd.DataFrame(random_search.cv_results_).query("rank_test_score == 1")
print(len(tied), "draws tie for first:", list(zip(tied["param_max_depth"], tied["param_min_samples_leaf"])))
print("best settings:", random_search.best_params_)
print("best mean score:", round(random_search.best_score_, 3))
print("test accuracy:", round(random_search.score(X_test, y_test), 3))

5 draws tie for first: [(5, 22), (10, 25), (9, 24), (7, 26), (6, 17)]
best settings: {'max_depth': 5, 'min_samples_leaf': 22}
best mean score: 0.828
test accuracy: 0.813


Five of the fifteen draws tie at 0.828, all with a leaf size from 17 to 26, and `best_params_` is the
first of them drawn. Fifteen draws over 10 depths and 39 leaf sizes came close to the grid's 0.83
while covering a far wider range.

### Choosing by the right score

The search chooses by accuracy unless `scoring` names another measure. For June frost, rare as the
**Imbalanced Classes** notebook showed, the choice between an unweighted and a weighted model depends
on it:

In [8]:
frost = network["june_frost"]
F_train, _, frost_train, _ = train_test_split(X, frost, test_size=0.25, random_state=0, stratify=frost)
weights = {"logisticregression__class_weight": [None, "balanced"]}

for scoring in [None, "recall"]:
    weight_search = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression()), weights, cv=folds,
                                 scoring=scoring).fit(F_train, frost_train)
    means = pd.DataFrame(weight_search.cv_results_)["mean_test_score"].round(3).tolist()
    print(f"scoring={scoring}: chose {weight_search.best_params_}, scores {means}")

scoring=None: chose {'logisticregression__class_weight': None}, scores [0.958, 0.889]
scoring=recall: chose {'logisticregression__class_weight': 'balanced'}, scores [0.336, 0.881]


By accuracy, the unweighted model wins, 0.958 to 0.889. By recall, the weighted one wins, 0.881 to
0.336. The search did exactly what it was told both times, and only `scoring` says what the search is
for.

### Nested cross-validation

`best_score_` is the best of several estimates, so it leans optimistic. Nested cross-validation puts
the whole search inside `cross_val_score`, so that each outer fold scores a winner chosen without it.
The lean is easiest to see where each estimate is noisy, so this runs on 150 of the training stations,
and the test set stays out of it:

In [9]:
small_X, small_y = X_train.head(150), y_train.head(150)
inner_folds = StratifiedKFold(5, shuffle=True, random_state=0)
inner = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=inner_folds)

nested = cross_val_score(inner, small_X, small_y, cv=StratifiedKFold(5, shuffle=True, random_state=1))
print("outer folds:", nested.round(3), "| mean:", round(nested.mean(), 3))

whole = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=inner_folds).fit(small_X, small_y)
print("best_score_ of one search on the same 150 stations:", round(whole.best_score_, 3))

outer folds: [0.733 0.733 0.8   0.767 0.767] | mean: 0.76
best_score_ of one search on the same 150 stations: 0.807


Each of the five outer folds ran a search of eighteen combinations, five folds each, on the other four
fifths, and scored its winner on the fold. The nested estimate is 0.76, and a single search on the
same 150 stations reports 0.807 for its winner. A single outer seed could be lucky or unlucky, so the
next cell repeats the nested estimate for eight seeds:

In [10]:
by_seed = [cross_val_score(inner, small_X, small_y, cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean()
           for seed in range(8)]

print("nested means for outer seeds 0 to 7:", np.round(by_seed, 3))
print("every one below best_score_:", all(mean < whole.best_score_ for mean in by_seed))

nested means for outer seeds 0 to 7: [0.733 0.76  0.76  0.747 0.793 0.767 0.773 0.76 ]
every one below best_score_: True


The nested estimate moves with the seed, from 0.733 to 0.793, and stays below 0.807 every time. That
gap is the amount by which choosing the best of eighteen noisy estimates flatters the choice.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/10-hyperparameter-search-solutions.ipynb).

**1.** Search `max_depth` from 2 to 8 alone, with `folds`, and print `best_params_` and `best_score_`.

In [11]:
# your code here

**2.** From `search.cv_results_`, the grid of depths and leaf sizes in the worked examples, print the mean score of every combination with `min_samples_leaf=20`, sorted by depth.

In [12]:
# your code here

**3.** Search the logistic pipeline's `C` over `[0.1, 1, 10]` with `scoring="f1"`, and print the winner.

In [13]:
# your code here

**4.** Run a `RandomizedSearchCV` of the tree with `n_iter=5` and `random_state=1`, and print its best settings and test accuracy.

In [14]:
# your code here

**5.** Estimate the logistic pipeline's search over `C` on the training stations with nested cross-validation, using `folds` inside and `StratifiedKFold(5, shuffle=True, random_state=1)` outside, and print the mean.

In [15]:
# your code here

## Common errors

### ValueError: Invalid parameter 'max_dept' for estimator DecisionTreeClassifier(random_state=0).

In [16]:
GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_dept": [2, 3]}, cv=folds).fit(X_train, y_train)

ValueError: Invalid parameter 'max_dept' for estimator DecisionTreeClassifier(random_state=0). Valid parameters are: ['ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].

A setting's name is checked when the search first fits, and a misspelled one stops it. The rest of the
message lists every valid name, `max_depth` among them. For a pipeline the names carry the step's
prefix, and `pipeline.get_params()` lists them.

### No error, and the best value at the edge of the grid

In [17]:
narrow = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression()),
                      {"logisticregression__C": [0.001, 0.003, 0.01]}, cv=folds).fit(X_train, y_train)

print("best settings:", narrow.best_params_)
print("mean scores:", pd.DataFrame(narrow.cv_results_)["mean_test_score"].round(3).tolist())

best settings: {'logisticregression__C': 0.01}
mean scores: [0.686, 0.811, 0.833]


The search chose 0.01, the largest value it was given, with the scores still rising toward it. The
wider grid in the worked examples found that `C=1` scores 0.854, against 0.833 here. A winner at the
edge says to extend the range in that direction and search again.

### No error, and a best score that flatters the winner

In [18]:
print("best_score_ on 150 stations:", round(whole.best_score_, 3))
print("nested estimates:", round(min(by_seed), 3), "to", round(max(by_seed), 3))

best_score_ on 150 stations: 0.807
nested estimates: 0.733 to 0.793


Reporting `best_score_` as the model's expected accuracy reports the luckiest of eighteen estimates,
here 0.807 against nested estimates from 0.733 to 0.793. The nested estimate is the number to report
for a model chosen by a search. A test set kept out of both the search and the nested run is the check
on it, scored once at the end.

## Recap

- Search every combination of settings with `GridSearchCV`, or a fixed number of draws with
  `RandomizedSearchCV`
- Read `best_params_`, `best_score_` and `cv_results_`, and predict with the refitted winner
- Name a pipeline's settings by the step, as in `logisticregression__C`
- Choose `scoring` to match the purpose, since a search maximizes only what it is given
- Estimate a searched model with nested cross-validation, and widen a grid whose winner is at its edge

## What is next

The **Feature Importance** notebook asks which features a fitted model relies on, and shows where the
built-in answer misleads.

---

&#8592; **Previous:** [Imbalanced Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/09-imbalanced-classes.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Feature Importance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/11-feature-importance.ipynb) &#8594;
